In [6]:
import pandas as pd
import numpy as np
import json

# Load the JSON
with open("sizing_data_full.json", "r") as f:
    SIZING = json.load(f)

# Example: access Aritzia size data
print(SIZING["Aritzia"]["core"]["alpha"]["bust"])

# Example: convert a size using your function
result = convert_size_least_squares(SIZING, "Aritzia", "S", "Zara", garment_type="tops")
print(result)


{'2XS': 30.7, 'XS': 32.6, 'S': 34.5, 'M': 36.5, 'L': 39.1, 'XL': 41.2, 'XXL': 44.2}
{'from': {'brand': 'Aritzia', 'size': 'S', 'garment_type': 'tops', 'vector': {'bust': 34.5, 'waist': 26.5, 'hip': 36.5}}, 'to': {'brand': 'Zara', 'suggestions': {'alpha': {'label': 'S', 'loss': 0.4375, 'dims_compared': ['bust', 'hip', 'waist'], 'rmse': 0.661}, 'numeric': {'label': '4', 'loss': 0.75, 'dims_compared': ['bust', 'hip', 'waist'], 'rmse': 0.866}}}}


In [4]:
from typing import Dict, List, Tuple, Optional
import math

# which dimensions to consider and their default weights by garment type
DIM_PRIORS = {
    "tops":    {"bust": 1.0, "waist": 0.5, "hip": 0.25, "shoulder": 0.4, "sleeve": 0.3, "length": 0.2},
    "bottoms": {"waist": 1.0, "hip": 0.8, "rise": 0.4, "inseam": 0.5, "length": 0.3},
    "dresses": {"bust": 0.8, "waist": 0.8, "hip": 0.8, "length": 0.2}
}

def _collect_size_vector(brand: Dict, label: str, dims: List[str]) -> Dict[str, float]:
    """Pick the value for each dim from alpha or numeric (prefers exact match); return only found dims."""
    out = {}
    core = brand.get("core", {})
    for dim in dims:
        v = None
        for sys in ("alpha", "numeric"):
            table = core.get(sys, {}).get(dim, {})
            if label in table:
                v = float(table[label])
                break
        if v is None:
            # try extras (e.g., jeans inseams, rises) if present
            ex = brand.get("extras", {})
            for section in ex.values():
                if isinstance(section, dict) and dim in section and isinstance(section[dim], dict) and label in section[dim]:
                    v = float(section[dim][label])
                    break
        if v is not None:
            out[dim] = v
    return out

def _candidate_vectors(brand: Dict, dims: List[str]) -> List[Tuple[str, Dict[str, float], str]]:
    """All candidate sizes in target brand: returns (label, vector, system) across alpha+numeric."""
    cands = []
    core = brand.get("core", {})
    for sys in ("alpha", "numeric"):
        # union of size labels across dims within this system
        labels = set()
        for dim in dims:
            labels |= set(core.get(sys, {}).get(dim, {}).keys())
        for lbl in labels:
            vec = {dim: float(core[sys][dim][lbl]) for dim in dims if lbl in core.get(sys, {}).get(dim, {})}
            if vec:
                cands.append((lbl, vec, sys))
    return cands

def convert_size_least_squares(
    sizing: Dict[str, Dict],
    from_brand: str,
    from_size: str,
    to_brand: str,
    garment_type: str = "tops",
    dim_overrides: Optional[Dict[str, float]] = None
) -> Dict:
    """
    Map a size label from one brand to another by minimizing weighted squared error across shared dims.
    - garment_type sets default weights; pass dim_overrides to tweak (e.g., {"hip": 1.2}).
    - Returns best alpha and numeric suggestions (if available), with residuals.
    """
    if from_brand not in sizing or to_brand not in sizing:
        return {"error": "Unknown brand(s)."}

    # pick dims and weights
    base = DIM_PRIORS.get(garment_type, DIM_PRIORS["tops"])
    weights = dict(base)
    if dim_overrides:
        weights.update(dim_overrides)
    dims = [d for d, w in weights.items() if w > 0]

    src_vec = _collect_size_vector(sizing[from_brand], from_size, dims)
    if not src_vec:
        return {"error": f"No usable measurements for {from_brand} size '{from_size}' in dims {dims}."}

    # candidates in target brand
    cand_list = _candidate_vectors(sizing[to_brand], dims)
    if not cand_list:
        return {"error": f"No candidate sizes for {to_brand} in dims {dims}."}

    def loss(vec_to: Dict[str, float]) -> float:
        # only compare on shared keys
        common = set(src_vec.keys()) & set(vec_to.keys())
        if not common:
            return math.inf
        return sum(weights[d] * (src_vec[d] - vec_to[d]) ** 2 for d in common)

    # evaluate per system for clarity
    best = {}
    for lbl, vec, sys in cand_list:
        L = loss(vec)
        if math.isinf(L): 
            continue
        if sys not in best or L < best[sys]["loss"]:
            best[sys] = {"label": lbl, "loss": L, "dims_compared": sorted(set(src_vec.keys()) & set(vec.keys()))}

    # prettify residuals
    for sys in list(best.keys()):
        best[sys]["rmse"] = round(math.sqrt(best[sys]["loss"]), 3)

    return {
        "from": {"brand": from_brand, "size": from_size, "garment_type": garment_type, "vector": {k: round(v,2) for k,v in src_vec.items()}},
        "to": {"brand": to_brand, "suggestions": best}
    }
